# BaF A²Π₁/₂ ← X²Σ⁺ — cooling-line branching & Table II frequency check

Standalone deliverable (generated by `make_baf_xa_cooling_nb.py`; **not** the
hand-built `BaF X-A.ipynb`). Constants: `¹³⁸Ba¹⁹F` boson X0/A0 in
`Source Code/molecule_parameters.py` (arXiv:2511.06986; B&C Eq.8.511/p.845 +
Steimle PRA 84 012508; A0 `Origin` = N² T₀,₀ + Be_A R² G-row). Spec:
`docs/superpowers/specs/2026-05-16-baf-xa-pipeline-design.md`.

Two things shown:
1. **Frequencies** — the three Table II components of the *(−)/N=0* line
   (the paper's precision-measured line), matched F_X→F_A line-by-line.
2. **Cooling branching** — the rotationally-closed laser-cooling transition
   **A²Π₁/₂(v=0, J=½, +) → X²Σ⁺(v=0, N=1)**. A `+`-parity decays E1 only to
   X `−`-parity (odd N=1); N=0/N=2 are `+`-parity, forbidden — the closure
   that makes it a cooling cycle.


In [ ]:
from config_path import add_to_sys_path
add_to_sys_path()  # walk up to "Source Code"

import numpy as np
from Energy_Levels import MoleculeLevels, branching_ratios

c = 29979.2458  # MHz per cm^-1 (== molecule_parameters.c)

def build(elec, N_list):
    return MoleculeLevels.initialize_state(
        molecule_name='BaF', elec_state=elec, vib_state=0,
        N_list=np.array(N_list), fermion_or_boson='boson',
        M_sublevels='none', I_nuclei=[0, 1/2], isotope=138,
        round=8, params=None, P_values=[1/2])

g = build('X', [0, 1])     # X2Sig+ v=0: N=0 (+par), N=1 (-par)
e = build('A', [1, 2])     # A2Pi1/2 v=0: J=1/2 (both Lambda-parities), J=3/2
g.eigensystem(0, 0)
e.eigensystem(0, 0)

def dom(s, i, q):
    """Dominant basis label of quantum q for eigenstate i."""
    return s.q_numbers[q][np.argmax(s.evecs0[i] ** 2)]

print("X q_str:", g.q_str)
print("A q_str:", e.q_str)


In [ ]:
# Hyperfine-splitting checks (Origin-independent; test the constant mappings)
gx0 = g.select_q({'N': 0, 'J': 0.5})              # X N=0,J=1/2  (F=0,1)
x_split = float(abs(np.diff(np.sort(g.evals0[gx0]))[-1]))
exm = e.select_q({'J': 0.5}, parity='-')          # A 2Pi1/2 J=1/2 (-) (F=0,1)
a_split = float(abs(np.diff(np.sort(e.evals0[exm]))[-1]))

print(f"X  N=0,J=1/2 F-split       = {x_split:8.3f} MHz   (Table II 65.6)")
print(f"A  2Pi1/2 J=1/2(-) F-split = {a_split:8.3f} MHz   (Table II 21.8)")
print("  -> Fermi-contact (X) and §3.3 a/d (A) mappings OK if both within ~1 MHz")


In [ ]:
# The 3 Table II (-)-line components, matched F_X -> F_A, line-by-line.
# nu = (E_A - E_X) + c*Origin   (E in MHz, Origin in cm^-1)
Org = e.parameters['Origin']
TBL2 = {(1, 0): 348666402.6, (1, 1): 348666424.4, (0, 1): 348666490.0}
print(f"A Origin = {Org:.6f} cm^-1\n")
print(f"{'F_X->F_A':>9} {'computed (MHz)':>16} {'Table II':>14} {'d':>7}")
for jx in gx0:
    Fx = int(dom(g, jx, 'F'))
    for ia in exm:
        Fa = int(dom(e, ia, 'F'))
        nu = (e.evals0[ia] - g.evals0[jx]) + c * Org
        ref = TBL2.get((Fx, Fa))
        if ref is None:
            print(f"   F{Fx}->F{Fa} {nu:16.1f} {'(E1-forbidden)':>14}")
            continue
        print(f"   F{Fx}->F{Fa} {nu:16.1f} {ref:14.1f} {nu-ref:+7.1f}")
print("\n(paper absolute accuracy ~1 MHz; constants are fixed literature, not refit)")


## Laser-cooling branching: A²Π₁/₂(v=0, J=½, **+**) → X²Σ⁺(v=0, **N=1**)

The cooling cycle. Parity selection makes it rotationally closed: the
`+`-parity A state decays E1 only to `−`-parity X levels = odd N = **N=1**
(N=0, N=2 are `+`-parity → forbidden). So branching into N=1 ≈ 1 and into
N=0 ≈ 0 is the *expected, required* structure (not a leak). The spread over
the four X N=1 hyperfine levels (J=½ F=0,1 and J=3/2 F=1,2) is the repump
structure a real BaF cooling experiment addresses. Magnitudes are idealized
single-state values (spec §3.6: the measured A²Π₁/₂–B²Σ⁺ perturbation is not
modelled — no quantitative intensity match expected).


In [ ]:
exp = e.select_q({'J': 0.5}, parity='+')               # cooling excited state
gN = np.array([dom(g, i, 'N') for i in range(g.size)]) # N per X eigenstate
BR = branching_ratios(Ground=g, Excited=e, Ez=0, Bz=0) # (nX, nA), unnormalized
assert np.all(np.isfinite(BR)) and np.all(BR >= 0), "BR not finite/non-negative"

n1 = [i for i in range(g.size) if dom(g, i, 'N') == 1]
hdr = "  ".join(f"N1[J={dom(g,i,'J')},F={int(dom(g,i,'F'))}]" for i in n1)
print(f"A(+) sublevel      ->X N=1   ->X N=0      {hdr}")
closed = True
for col in exp:
    p = BR[:, col] / BR[:, col].sum()
    to_N0, to_N1 = float(p[gN == 0].sum()), float(p[gN == 1].sum())
    per = "  ".join(f"{p[i]:13.4f}" for i in n1)
    print(f"  idx {col} (F={int(dom(e,col,'F'))})  {to_N1:9.4f}  {to_N0:8.1e}     {per}")
    closed &= (to_N1 >= 0.999) and (to_N0 <= 1e-3) and abs(p.sum()-1) <= 1e-6
print("\nrotational closure into X N=1:", "OK" if closed else "CHECK")


## Summary

- **Frequencies:** the three Table II (−)-line components reproduced
  line-by-line (≈ +0.6 / +0.8 / +1.4 MHz at the time of validation), within
  the paper's ~1 MHz absolute accuracy using fixed (non-refit) constants.
- **Splittings:** X N=0 ≈ 66.25 (Table II 65.6) and A²Π₁/₂(−) ≈ 21.93
  (Table II 21.8) — the Fermi-contact and §3.3 a/d mappings.
- **Cooling BR:** A²Π₁/₂(J=½,+) → X N=1 is parity-closed (Σ→N=1 ≈ 1,
  Σ→N=0 ≈ 0) and spreads over the X N=1 hyperfine manifold.

Reproducible: regenerate with `python make_baf_xa_cooling_nb.py`. The
standalone assertion form is `baf_xa_validate.py`.
